## 5.5 功率自适应仿真

在上一节中，我们学习了功率控制的三类信令与四种场景。本节通过仿真演示闭环功率自适应的完整过程：在较差信道（SNR=0 dB）下启动，连续 CRC 失败触发射功率上调，观察 SNR 的提升与 FER 的改善。

本节学习大纲如下：

- 功率自适应闭环逻辑
- 功率轨迹与成功/失败散点图
- 升功率对 FER 的改善效果

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── node.py                  <- SleNode: adjust_power / process_feedback
├── mac/
│   ├── power_control.py     <- PowerController: 功率状态管理
│   │                           PowerControlRequest / Response / ChangeIndication
│   └── frame.py             <- AsyncDataFrame: MAC 数据帧
├── phy/
│   └── channel.py           <- ChannelModel: AWGN 信道
└── sim/
    └── link_sim.py          <- sim_node_power_adapt: 功率自适应仿真
```

---

### 1. 功率自适应原理

实验要点：
1. 初始以较低功率发射（节省功耗、减少干扰）
2. 监控 CRC 反馈：连续 $N$ 次失败 → 升功率 $\Delta P$ dB
3. 成功 → 重置失败计数器
4. 功率钳位在 [min_power, max_power] 范围内


可运行以下代码查看仿真源码：

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "4285,4366p"

---

### 2. 功率自适应仿真

初始 SNR=0 dB，初始功率=0 dBm，连续 3 次失败升 2 dB，上限 15 dBm：

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode

# ===== 功率自适应参数 =====
base_snr_db = 0.0                
max_power = 15.0                  # 最大发射功率 dBm 
min_power = -10.0                 # 最小发射功率 dBm 
power_step = 2.0               
n_frames = 80                     
threshold = 3                     # 连续失败阈值: 连续失败 3 次触发升功率

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"
rng = np.random.default_rng(42)

# ===== 建链: G 广播 → T 扫描 → T 连接 → G 接受 =====
g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7,                              # MCS=7: QPSK 7/8, 频谱效率 1.75
    max_retransmit=0,
    tx_power_dbm=0.0,                                       
    max_power_dbm=max_power, min_power_dbm=min_power))     
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0))
g.start_advertising(); t.start_scanning()
t.connect(g_addr); g.accept_connection(t_addr, Role.G_NODE)

# ===== 逐帧仿真 =====
power_hist = []                   # 每帧的发射功率记录
success_hist = []                 # 每帧的 CRC 结果
consec_fails = 0                  # 连续失败计数器
tx_power = 0.0                    # 当前发射功率 (dBm)

for i in range(n_frames):
    # ---- 发送端: 打包数据 → 生成 IQ ----
    payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    g.send(payload)                                          # 数据压入发送队列
    tx = g.transmit()                                        # MAC-PHY 转换 → IQ 信号
    if tx.iq is None:                                        # IQ 生成失败 (如队列空)
        power_hist.append(tx_power)
        success_hist.append(False)
        continue

    # ---- 功率缩放: dBm → 线性增益 → 放大 IQ 幅值 ----
    gain = 10.0 ** (tx_power / 20.0)                         
    scaled = tx.iq * gain                                    

    # ---- AWGN 信道 ----
    eff_snr = base_snr_db + tx_power                         # SNR_eff = SNR_base + P_tx (dB)
    snr_lin = 10.0 ** (eff_snr / 10.0)                       
    noise_std = np.sqrt(1.0 / (2.0 * snr_lin))               
    noise = noise_std * (rng.standard_normal(len(scaled))    
                       + 1j * rng.standard_normal(len(scaled)))  
    noisy = scaled + noise                                    

    # ---- 接收端: 解调 + CRC 校验 ----
    frame = AsyncDataFrame(segment_type=0, data=payload)     
    rx = t.receive(noisy, len(frame.pack()))                 # T 接收: 匹配滤波 → 解调 → CRC
    success = rx.success and rx.data == payload              

    # ---- ARQ 反馈: 通知 G 本帧结果 ----
    g.process_feedback(rx.success)
    if not success:
        g._qos.arq.on_ack_received()                         # 失败时强制推进 ARQ 状态机

    # ===== 功率自适应核心逻辑 =====
    if not success:
        consec_fails += 1                                    # 失败: 连续失败 +1
        if consec_fails >= threshold:                        # 达到阈值 → 触发升功率
            tx_power = min(tx_power + power_step, max_power) # 升 power_step dB
            consec_fails = 0                                 
    else:
        consec_fails = 0                                     

    power_hist.append(tx_power)                             
    success_hist.append(success)                            

---

### 3. 功率轨迹与成功/失败分布

In [ ]:
fer = 1.0 - sum(success_hist) / len(success_hist)
ok_idx = [i for i, s in enumerate(success_hist) if s]
fail_idx = [i for i, s in enumerate(success_hist) if not s]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Power Adaptation (base SNR={base_snr_db:.0f} dB, threshold={threshold})", fontsize=13)

ax1.plot(power_hist, "b-", lw=1.5, label="TX Power")
ax1.set_xlabel("Frame Index"); ax1.set_ylabel("TX Power (dBm)")
ax1.set_title(f"Power Trajectory (FER={fer:.3f})")
ax1.legend(); ax1.grid(True, ls="--", alpha=0.5)

ax2.scatter(ok_idx, [1]*len(ok_idx), c="green", s=8, alpha=0.5, label="OK")
ax2.scatter(fail_idx, [0]*len(fail_idx), c="red", s=8, alpha=0.5, label="FAIL")
ax2.set_xlabel("Frame Index")
ax2.set_yticks([0, 1]); ax2.set_yticklabels(["FAIL", "OK"])
ax2.set_title("Frame Success/Failure")
ax2.legend(loc="upper right"); ax2.grid(True, ls="--", alpha=0.3)
plt.tight_layout(); plt.show()

print(f"FER = {fer:.3f}")
print(f"Power range: {min(power_hist):.0f} -> {max(power_hist):.0f} dBm")
print(f"SNR range:  {base_snr_db + min(power_hist):.0f} -> {base_snr_db + max(power_hist):.0f} dB")
print(f"Fail frames: {fail_idx}")

---

### 4. 实验分析

根据上图可以观察：

- **左图（功率轨迹）**：观察发射功率如何随帧索引变化——初始维持在什么水平，经过多少帧后开始上升，最终收敛到哪个功率值。思考功率阶梯的宽度与连续失败阈值的关系。
- **右图（成功/失败分布）**：观察红色失败点和绿色成功点的分布密度如何随功率上升而变化。功率较低的帧是否集中在红色区域，功率上升后绿色点是否逐渐占据主导。
- **综合**：对比两图的时间轴，思考功率自适应的响应速度（升得太慢会怎样？升得太快会怎样？），以及阈值和步长这两个参数对 FER 和功率收敛值的权衡影响。

---

## 课后实践

请补全下方功率自适应核心逻辑中的 **3 处空缺**（每处一行代码），完成连续失败计数→阈值触发→升功率的闭环控制。

要求：

1. 补全失败时的计数器更新
2. 补全触发升功率的阈值判断条件
3. 补全升功率操作（含上限钳位）

完成后运行 `python power_practice.py`，观察功率轨迹是否呈阶梯上升。

In [ ]:
%%writefile power_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode

base_snr_db = 0.0
max_power = 15.0
power_step = 2.0
n_frames = 60
threshold = 3

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"
rng = np.random.default_rng(42)

g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0,
    tx_power_dbm=0.0, max_power_dbm=max_power, min_power_dbm=-10.0))
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0))
g.start_advertising(); t.start_scanning()
t.connect(g_addr); g.accept_connection(t_addr, Role.G_NODE)

consec_fails, tx_power = 0, 0.0
power_hist = []

for i in range(n_frames):
    payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    g.send(payload)
    tx = g.transmit()
    if tx.iq is None:
        power_hist.append(tx_power); continue
    # 功率缩放 + AWGN + 接收
    gain = 10.0 ** (tx_power / 20.0)
    scaled = tx.iq * gain
    eff_snr = base_snr_db + tx_power
    snr_lin = 10.0 ** (eff_snr / 10.0)
    noise_std = np.sqrt(1.0 / (2.0 * snr_lin))
    noise = noise_std * (rng.standard_normal(len(scaled)) + 1j * rng.standard_normal(len(scaled)))
    noisy = scaled + noise
    frame = AsyncDataFrame(segment_type=0, data=payload)
    rx = t.receive(noisy, len(frame.pack()))
    success = rx.success and rx.data == payload
    g.process_feedback(rx.success)
    if not success:
        g._qos.arq.on_ack_received()

    # ==== 补全功率自适应核心逻辑（3处空缺）====
    if not success:
        ______________  # 1: 连续失败计数 +1 （补全）
        ______________  # 2: 达到阈值触发升功率（补全）
            ______________  # 3: 升功率 + 上限钳位（补全）
            consec_fails = 0
    else:
        consec_fails = 0

    power_hist.append(tx_power)

print(f"Final power: {power_hist[-1]:.0f} dBm")
print(f"Power stepped: {len(set(power_hist))} distinct levels")
print("PASS" if power_hist[-1] >= 4.0 else "check logic — power should rise")


执行以下命令进行编译并验证结果：


In [ ]:
!python power_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/05.05_answer.txt
